# Customer Lifetime Value (CLTV) Analysis

## Objective

The objective of this notebook is to estimate the historical Customer Lifetime Value (CLTV) of customers using transactional purchasing behavior.

The analysis includes:

- Revenue feature engineering
- Customer purchase frequency
- Average order value
- Customer lifespan estimation
- Historical CLTV calculation
- Customer segmentation
- Executive visualizations

The outputs generated in this notebook will support executive reporting and Power BI dashboard development.

In [2]:
# ==========================================
# Import Libraries
# ==========================================

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

pd.set_option("display.float_format", "{:,.2f}".format)

In [3]:
# ==========================================
# Project Paths
# ==========================================

PROJECT_ROOT = Path.cwd().parent

RAW_DATA = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA = PROJECT_ROOT / "data" / "processed"
VISUALS = PROJECT_ROOT / "visuals"

In [4]:
# ==========================================
# Load Clean Dataset
# ==========================================

df = pd.read_csv(
    PROCESSED_DATA / "online_retail_clean.csv",
    parse_dates=["InvoiceDate"]
)

print(df.shape)
df.head()

(779425, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,"13,085.00",United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,"13,085.00",United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,"13,085.00",United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,"13,085.00",United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,"13,085.00",United Kingdom


In [5]:
# ==========================================
# Data Validation
# ==========================================

print("Shape:", df.shape)
print("Missing Customer IDs:", df["Customer ID"].isna().sum())
print("Negative Quantity:", (df["Quantity"] < 0).sum())
print("Invalid Prices:", (df["Price"] <= 0).sum())
print("Cancelled Invoices:", df["Invoice"].astype(str).str.startswith("C").sum())

Shape: (779425, 8)
Missing Customer IDs: 0
Negative Quantity: 0
Invalid Prices: 0
Cancelled Invoices: 0


In [6]:
# ==========================================
# Create Revenue Feature
# ==========================================

df["Revenue"] = df["Quantity"] * df["Price"]

df[["Quantity", "Price", "Revenue"]].head()

,Quantity,Price,Revenue
0,12,6.95,83.40
1,12,6.75,81.00
2,12,6.75,81.00
3,48,2.10,100.80
4,24,1.25,30.00


In [7]:
# ==========================================
# Revenue Summary
# ==========================================

df["Revenue"].describe().round(2)

count   779,425.00
mean         22.29
std         227.43
min           0.00
25%           4.95
50%          12.48
75%          19.80
max     168,469.60
Name: Revenue, dtype: float64

In [8]:
# ==========================================
# Customer-Level Summary
# ==========================================

customer_summary = (
    df.groupby("Customer ID")
      .agg(
          TotalRevenue=("Revenue", "sum"),
          TotalOrders=("Invoice", "nunique"),
          TotalTransactions=("Invoice", "count"),
          TotalProducts=("Quantity", "sum"),
          FirstPurchase=("InvoiceDate", "min"),
          LastPurchase=("InvoiceDate", "max")
      )
      .reset_index()
)

customer_summary.head()

,Customer ID,TotalRevenue,TotalOrders,TotalTransactions,TotalProducts,FirstPurchase,LastPurchase
0,"12,346.00","77,556.46",12,34,74285,2009-12-14 08:34:00,2011-01-18 10:01:00
1,"12,347.00","4,921.53",8,222,2967,2010-10-31 14:20:00,2011-12-07 15:52:00
2,"12,348.00","2,019.40",5,51,2714,2010-09-27 14:59:00,2011-09-25 13:13:00
3,"12,349.00","4,428.69",4,175,1624,2010-04-29 13:20:00,2011-11-21 09:51:00
4,"12,350.00",334.40,1,17,197,2011-02-02 16:01:00,2011-02-02 16:01:00


In [9]:
print(customer_summary.shape)

customer_summary.describe().round(2)

(5878, 7)


,Customer ID,TotalRevenue,TotalOrders,TotalTransactions,TotalProducts,FirstPurchase,LastPurchase
count,"5,878.00","5,878.00","5,878.00","5,878.00","5,878.00",5878,5878
mean,"15,315.31","2,955.90",6.29,132.60,"1,788.70",2010-08-22 06:57:33.297039,2011-05-22 16:19:59.469207
min,"12,346.00",2.95,1.00,1.00,1.00,2009-12-01 07:45:00,2009-12-01 09:55:00
25%,"13,833.25",342.28,1.00,20.00,187.00,2010-02-09 14:01:15,2010-11-25 10:24:45
50%,"15,314.50",867.74,3.00,52.00,480.00,2010-06-27 13:31:30,2011-09-05 11:59:00
75%,"16,797.75","2,248.30",7.00,138.00,"1,350.00",2011-01-30 14:30:15,2011-11-14 11:31:15
max,"18,287.00","580,987.04",398.00,"12,435.00","367,193.00",2011-12-09 12:16:00,2011-12-09 12:50:00
std,"1,715.57","14,440.85",13.01,342.19,"8,876.30",NaN,NaN


## Customer-Level Summary

A customer-level analytical dataset was created by aggregating individual transaction records into one record per customer. The summary includes total revenue, order count, transaction volume, quantity of products purchased, and purchase history.

### Key Findings

- The dataset contains **5,878 unique customers**.
- Average customer revenue is **2,955.90**, while the median revenue is **867.74**, indicating a highly right-skewed revenue distribution.
- Customer purchasing behavior varies substantially, with orders ranging from **1** to **398**.
- A small number of customers contribute exceptionally high revenues, suggesting the presence of high-value or wholesale customers.
- The customer summary provides the analytical foundation for calculating Average Order Value (AOV), Purchase Frequency, Customer Lifespan, and Historical Customer Lifetime Value (CLTV).

In [10]:
# ==========================================
# Calculate Customer Lifespan
# ==========================================

customer_summary["CustomerLifespan"] = (
    customer_summary["LastPurchase"] -
    customer_summary["FirstPurchase"]
).dt.days

customer_summary["CustomerLifespan"].describe().round(2)

count   5,878.00
mean      273.02
std       258.81
min         0.00
25%         0.00
50%       220.50
75%       511.00
max       738.00
Name: CustomerLifespan, dtype: float64

## Customer Lifespan Analysis

Customer lifespan was calculated as the number of days between each customer's first and last recorded purchases.

### Key Findings

- The average customer lifespan is **273 days** (approximately nine months).
- The median lifespan is **220 days**, indicating that half of the customers remained active for more than seven months.
- Approximately **25% of customers made only one purchase**, resulting in a lifespan of zero days.
- The longest customer relationship lasted **738 days**, demonstrating strong long-term customer retention for a subset of customers.

These findings highlight significant differences in customer loyalty and purchasing behavior, emphasizing the importance of Customer Lifetime Value (CLTV) analysis for identifying high-value, long-term customers.

In [11]:
customer_summary[
    ["FirstPurchase", "LastPurchase", "CustomerLifespan"]
].head()

,FirstPurchase,LastPurchase,CustomerLifespan
0,2009-12-14 08:34:00,2011-01-18 10:01:00,400
1,2010-10-31 14:20:00,2011-12-07 15:52:00,402
2,2010-09-27 14:59:00,2011-09-25 13:13:00,362
3,2010-04-29 13:20:00,2011-11-21 09:51:00,570
4,2011-02-02 16:01:00,2011-02-02 16:01:00,0


## Customer Lifespan Calculation

Customer lifespan was calculated as the number of days between each customer's first and last recorded purchases.

### Observations

- Customers who made repeat purchases have positive lifespan values, reflecting the duration of their relationship with the business.
- Customers with a lifespan of **0 days** are one-time buyers whose first and last purchases occurred on the same day.
- The calculated lifespan will be used in the Historical Customer Lifetime Value (CLTV) model to estimate long-term customer value.

In [12]:
# ==========================================
# Average Order Value
# ==========================================

customer_summary["AverageOrderValue"] = (
    customer_summary["TotalRevenue"] /
    customer_summary["TotalOrders"]
)

customer_summary["AverageOrderValue"].describe().round(2)

count    5,878.00
mean       385.18
std      1,214.29
min          2.95
25%        176.68
50%        279.24
75%        414.90
max     84,236.25
Name: AverageOrderValue, dtype: float64

In [18]:
customer_summary[
    [
        "Customer ID",
        "TotalRevenue",
        "TotalOrders",
        "AverageOrderValue"
    ]
].head()

,Customer ID,TotalRevenue,TotalOrders,AverageOrderValue
0,"12,346.00","77,556.46",12,"6,463.04"
1,"12,347.00","4,921.53",8,615.19
2,"12,348.00","2,019.40",5,403.88
3,"12,349.00","4,428.69",4,"1,107.17"
4,"12,350.00",334.40,1,334.40


In [13]:
total_customers = customer_summary.shape[0]

print(total_customers)

5878


In [14]:
# ==========================================
# Purchase Frequency
# ==========================================

customer_summary["PurchaseFrequency"] = (
    customer_summary["TotalOrders"] /
    total_customers
)

customer_summary["PurchaseFrequency"].describe().round(4)

count   5,878.00
mean        0.00
std         0.00
min         0.00
25%         0.00
50%         0.00
75%         0.00
max         0.07
Name: PurchaseFrequency, dtype: float64

In [19]:
customer_summary[
    [
        "Customer ID",
        "TotalOrders",
        "PurchaseFrequency"
    ]
].head()

,Customer ID,TotalOrders,PurchaseFrequency
0,"12,346.00",12,0.00
1,"12,347.00",8,0.00
2,"12,348.00",5,0.00
3,"12,349.00",4,0.00
4,"12,350.00",1,0.00


In [15]:
# ==========================================
# Customer Age
# ==========================================

snapshot_date = df["InvoiceDate"].max()

customer_summary["CustomerAge"] = (
    snapshot_date -
    customer_summary["FirstPurchase"]
).dt.days

customer_summary["CustomerAge"].describe().round(2)

count   5,878.00
mean      473.71
std       223.10
min         0.00
25%       312.00
50%       529.00
75%       667.00
max       738.00
Name: CustomerAge, dtype: float64

In [20]:
customer_summary[
    [
        "Customer ID",
        "FirstPurchase",
        "CustomerAge"
    ]
].head()

,Customer ID,FirstPurchase,CustomerAge
0,"12,346.00",2009-12-14 08:34:00,725
1,"12,347.00",2010-10-31 14:20:00,403
2,"12,348.00",2010-09-27 14:59:00,437
3,"12,349.00",2010-04-29 13:20:00,588
4,"12,350.00",2011-02-02 16:01:00,309


In [16]:
# ==========================================
# Historical CLTV
# ==========================================

customer_summary["HistoricalCLTV"] = (
    customer_summary["AverageOrderValue"] *
    customer_summary["PurchaseFrequency"] *
    customer_summary["CustomerLifespan"]
)

customer_summary["HistoricalCLTV"].describe().round(2)

count    5,878.00
mean       272.51
std      1,723.37
min          0.00
25%          0.00
50%         30.17
75%        162.62
max     72,944.61
Name: HistoricalCLTV, dtype: float64

In [17]:
customer_summary[
    [
        "AverageOrderValue",
        "PurchaseFrequency",
        "CustomerLifespan",
        "CustomerAge",
        "HistoricalCLTV"
    ]
].head()

,AverageOrderValue,PurchaseFrequency,CustomerLifespan,CustomerAge,HistoricalCLTV
0,"6,463.04",0.00,400,725,"5,277.74"
1,615.19,0.00,402,403,336.59
2,403.88,0.00,362,437,124.37
3,"1,107.17",0.00,570,588,429.46
4,334.40,0.00,0,309,0.00


## Customer Value Metrics

Customer Lifetime Value (CLTV) metrics were calculated using customer purchase history and transaction data.

The following metrics were derived:

- **Average Order Value (AOV):** Average revenue generated per customer order.
- **Purchase Frequency:** Relative purchasing activity across the customer base.
- **Customer Lifespan:** Number of days between a customer's first and last purchase.
- **Customer Age:** Number of days from the customer's first purchase until the latest transaction in the dataset.
- **Historical CLTV:** Estimated customer lifetime value calculated as:

> Historical CLTV = Average Order Value × Purchase Frequency × Customer Lifespan

These metrics quantify customer value and provide the foundation for customer segmentation and strategic decision-making.